# RF로 DDoS 탐지: 전체 vs 카메라만 비교

**하는 일**
1. 드라이브의 pcap을 직접 읽어서, 패킷 10개씩 묶은 window마다 통계 feature를 만들어 (MAC으로 기기 종류도 붙임)
2. Random Forest로 DDoS 탐지
3. ① 전체 데이터로 학습·평가  ② 카메라 window만 따로 학습·평가 → 정확도와 속도 비교

**쓰는 법**: 위에서부터 셀을 하나씩 ▶ 실행해. 바꿀 건 1번 셀의 설정뿐이야.
(1번에서 드라이브 연결 팝업이 뜨면 허용)

## 1) 설정 + 드라이브 연결

In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# ===== 설정: 여기만 바꾸면 돼 =====
PCAP_DIR = Path('/content/drive/MyDrive/CICIoT2023')   # pcap 넣어둔 폴더
MAX_PKTS = 2_000_000    # 파일 하나당 "앞에서부터" 읽을 패킷 수 (줄이면 빨라짐 / None이면 파일 전체 = 아주 오래 걸림)
WINDOW = 10             # 패킷 몇 개를 한 묶음(window)으로 볼지
TARGET_CAT = 'Camera'   # 따로 뽑아볼 기기 종류: Power Outlet / Camera / Home Automation / Lighting / Audio / Hub
MAX_TRAIN = 300_000     # RF 학습에 쓸 최대 window 수 (너무 크면 오래 걸림)
MAX_TEST = 200_000      # 평가에 쓸 최대 window 수
N_TREES = 50            # RF 트리 개수
MAX_DEPTH = 20          # RF 트리 최대 깊이

Mounted at /content/drive


## 2) 기기 MAC 표 (MAC → 기기 종류)

In [2]:
import numpy as np

# 논문 표(기기 MAC → 종류). 아까 분석한 것과 같은 표야.
DEVICES = """
1C:FE:2B:98:16:DD,Audio,Echo Dot 1
A0:D0:DC:C4:08:FF,Audio,Echo Dot 2
1C:12:B0:9B:0C:EC,Audio,Echo Spot
08:7C:39:CE:6E:2A,Audio,Echo Studio
2C:71:FF:05:F1:15,Audio,Echo Show
CC:F4:11:9C:D0:00,Audio,Google Nest Mini
B0:F1:EC:D3:E7:98,Audio,Harman Kardon
48:A6:B8:F9:1B:88,Audio,Sonos One
9C:8E:CD:1D:AB:9F,Camera,Amcrest
3C:37:86:6F:B9:51,Camera,Arlo Base Station
40:5D:82:35:14:C8,Camera,Arlo Q Indoor
C0:E7:BF:0A:79:D1,Camera,Borun/Sichuan-AI
B0:C5:54:59:2E:99,Camera,D-Link Mini
44:01:BB:EC:10:4A,Camera,HeimVision WiFi
34:75:63:73:F3:36,Camera,Home Eye
7C:A7:B0:CD:18:32,Camera,Luohe Cam Dog
44:BB:3B:00:39:07,Camera,Nest Indoor
70:EE:50:68:0E:32,Camera,Netatmo Camera
10:5A:17:97:A5:C6,Camera,Rbcior
10:2C:6B:1B:43:BE,Camera,SIMCAM 1S
6C:5A:B0:44:1D:90,Camera,TP-Link Tapo
7C:78:B2:86:0D:81,Camera,Wyze
84:7A:B6:64:62:58,Camera,Yi Indoor
84:7A:B6:62:3A:6C,Camera,Yi Indoor 2
2C:D2:6B:66:D2:87,Camera,Yi Outdoor
D4:A6:51:76:06:64,Power Outlet,Teckin Plug 1
D4:A6:51:78:97:4E,Power Outlet,Teckin Plug 2
30:23:03:F3:84:2B,Power Outlet,Wemo Plug 1
30:23:03:F3:57:CB,Power Outlet,Wemo Plug 2
D4:A6:51:20:91:D1,Power Outlet,Yutron Plug 1
D4:A6:51:21:6C:29,Power Outlet,Yutron Plug 2
B8:5F:98:D0:76:E6,Power Outlet,Amazon Plug
50:02:91:1A:CE:E1,Power Outlet,Gosund Power Strip 1
B8:F0:09:03:9A:AF,Power Outlet,Gosund Power Strip 2
B8:F0:09:03:29:79,Power Outlet,Gosund WP2 1
50:02:91:10:AC:D8,Power Outlet,Gosund WP2 2
50:02:91:10:09:8F,Power Outlet,Gosund WP2 3
C4:DD:57:0C:39:94,Power Outlet,Gosund WP3 1
24:A1:60:14:7F:F9,Power Outlet,Gosund WP3 2
84:E3:42:42:ED:0B,Lighting,Lumiman Bulb
18:69:D8:EB:D4:3E,Lighting,Teckin Light Strip
50:02:91:B1:68:0C,Lighting,Globe Lamp
C4:DD:57:13:07:C6,Lighting,Gosund Bulb
D4:A6:51:30:64:B7,Lighting,HeimVision Lamp
F4:CF:A2:34:48:6B,Lighting,LampUX RGB
D0:73:D5:35:FB:C8,Lighting,LIFX Bulb
00:02:75:F6:E3:CB,Home Automation,Smart Board
AC:F1:08:4E:00:82,Home Automation,LG Smart TV
70:EE:50:6B:A8:1A,Home Automation,Netatmo Weather Station
68:57:2D:56:AC:47,Home Automation,Atomi Coffee Maker
08:3A:F2:1F:BC:68,Home Automation,Cocoon HVAC Fan
D4:AD:FC:29:C8:A2,Home Automation,Govee Humidifier
50:14:79:37:80:18,Home Automation,iRobot Roomba
1C:9D:C2:8C:9A:94,Home Automation,Levoit Air Purifier
00:17:88:60:D6:4F,Hub,Philips Hue Bridge
8C:85:80:6C:B6:47,Hub,Eufy HomeBase 2
28:6D:97:7A:2B:2D,Hub,SmartThings Hub
28:6D:97:9E:F4:D5,Hub,AeoTec Hub
B0:09:DA:3E:82:6C,Hub,Ring Base Station
AC:17:02:05:34:27,Hub,Fibaro Home Center Lite
DC:A6:32:C9:E6:F4,NextGen,Raspberry Pi
DC:A6:32:C9:E4:C6,NextGen,Raspberry Pi
DC:A6:32:C9:E5:02,NextGen,Raspberry Pi
E4:5F:01:55:90:C4,NextGen,Raspberry Pi
DC:A6:32:C9:E4:D5,NextGen,Raspberry Pi
DC:A6:32:DC:27:D5,NextGen,Raspberry Pi
DC:A6:32:C9:E5:EF,NextGen,Raspberry Pi
DC:A6:32:C9:E4:AB,NextGen,Raspberry Pi
DC:A6:32:C9:E4:90,NextGen,Raspberry Pi
DC:A6:32:C9:E5:A4,NextGen,Raspberry Pi
"""

CATS = ['Unknown', 'Power Outlet', 'Camera', 'Home Automation', 'Lighting', 'Audio', 'Hub', 'NextGen']
CAT_CODE = {c: i for i, c in enumerate(CATS)}
NEXTGEN = CAT_CODE['NextGen']

_m = {}
for line in DEVICES.strip().splitlines():
    mac, cat, name = line.split(',')
    _m[int(mac.replace(':', ''), 16)] = CAT_CODE[cat]
KEYS = np.array(sorted(_m), dtype=np.int64)
CODES = np.array([_m[k] for k in KEYS], dtype=np.int8)

def mac_to_cat(macs):
    """MAC(정수 배열) → 기기 종류 코드. 표에 없으면 0(Unknown)"""
    idx = np.clip(np.searchsorted(KEYS, macs), 0, len(KEYS) - 1)
    return np.where(KEYS[idx] == macs, CODES[idx], 0)

print('등록된 MAC:', len(KEYS), '개')

등록된 MAC: 70 개


## 3) pcap 읽는 함수 준비 (실행만 하면 돼)

In [3]:
import struct, time
import numpy as np
import pandas as pd

POW6 = np.array([2**40, 2**32, 2**24, 2**16, 2**8, 1], dtype=np.int64)

def _extract(buf, offs, dt4):
    """buf 안 패킷들의 헤더 정보를 numpy로 한꺼번에 뽑는다 (offs = 각 패킷 레코드 시작 위치)"""
    arr = np.frombuffer(buf + b'\x00' * 128, dtype=np.uint8)
    ar = np.arange

    def take(base, n):                      # 패킷마다 base 위치부터 n바이트 (N x n)
        return arr[base[:, None] + ar(n)]

    def be16(base):                         # 2바이트 정수(big-endian)
        b = take(base, 2).astype(np.int64)
        return (b[:, 0] << 8) | b[:, 1]

    d = offs + 16                           # 이더넷 헤더 시작 위치
    sec = take(offs, 4).view(dt4).ravel().astype(np.int64)
    frac = take(offs + 4, 4).view(dt4).ravel().astype(np.int64)
    incl = take(offs + 8, 4).view(dt4).ravel().astype(np.int64)
    orig = take(offs + 12, 4).view(dt4).ravel().astype(np.int64)

    mac = take(d, 12).astype(np.int64)
    dst_mac = mac[:, :6] @ POW6
    src_mac = mac[:, 6:] @ POW6

    et = be16(d + 12)                       # 이더넷 타입
    is_arp = et == 0x0806
    is_ip6 = et == 0x86DD
    b14 = arr[d + 14].astype(np.int64)
    ihl = (b14 & 15) * 4                    # IP 헤더 길이
    is_ip4 = (et == 0x0800) & (incl >= 34) & ((b14 >> 4) == 4) & (ihl >= 20)

    ttl = np.where(is_ip4, arr[d + 22], 0)
    proto = np.where(is_ip4, arr[d + 23], 0)
    ff = be16(d + 20)                       # 플래그 + fragment offset
    off_nz = (ff & 0x1FFF) != 0
    is_frag = is_ip4 & (off_nz | ((ff & 0x2000) != 0))
    src_ip = np.where(is_ip4, take(d + 26, 4).view('>u4').ravel().astype(np.int64), 0)
    dst_ip = np.where(is_ip4, take(d + 30, 4).view('>u4').ravel().astype(np.int64), 0)

    l4 = d + 14 + ihl                       # TCP/UDP 헤더 시작 위치
    l4_ok = is_ip4 & ~off_nz
    is_tcp = l4_ok & (proto == 6) & (incl >= 14 + ihl + 20)
    is_udp = l4_ok & (proto == 17) & (incl >= 14 + ihl + 8)
    is_icmp = is_ip4 & (proto == 1)
    has_port = is_tcp | is_udp
    sport = np.where(has_port, be16(l4), 0)
    dport = np.where(has_port, be16(l4 + 2), 0)
    flags = np.where(is_tcp, arr[l4 + 13], 0)

    return dict(
        sec=sec, frac=frac, length=orig.astype(np.float32),
        dst_mac=dst_mac, src_mac=src_mac,
        ttl=ttl.astype(np.int16), proto=proto.astype(np.int16),
        is_tcp=is_tcp, is_udp=is_udp, is_icmp=is_icmp, is_arp=is_arp, is_ip6=is_ip6, is_frag=is_frag,
        src_ip=src_ip.astype(np.uint32), dst_ip=dst_ip.astype(np.uint32),
        sport=sport.astype(np.int32), dport=dport.astype(np.int32), flags=flags.astype(np.int16),
    )


def read_pcap_fields(path, max_pkts=None, chunk_bytes=64 * 1024 * 1024):
    """pcap 파일 앞에서부터 max_pkts개 패킷의 헤더 정보를 읽어 numpy 배열 dict로 돌려준다"""
    max_pkts = max_pkts or 10**12
    pieces, total, carry = [], 0, b''
    with open(path, 'rb') as f:
        gh = f.read(24)
        magic = gh[:4]
        if magic == b'\xd4\xc3\xb2\xa1':   end, div = '<', 1e6
        elif magic == b'\x4d\x3c\xb2\xa1': end, div = '<', 1e9
        elif magic == b'\xa1\xb2\xc3\xd4': end, div = '>', 1e6
        elif magic == b'\xa1\xb2\x3c\x4d': end, div = '>', 1e9
        else:
            raise ValueError('일반 pcap 형식이 아니에요(pcapng일 수 있음): ' + str(path))
        if struct.unpack(end + 'I', gh[20:24])[0] != 1:
            raise ValueError('이더넷(link type 1) pcap이 아니에요: ' + str(path))
        dt4 = np.dtype(end + 'u4')
        u4 = struct.Struct(end + 'I').unpack_from

        while total < max_pkts:
            data = f.read(chunk_bytes)
            if not data:
                break
            buf = carry + data
            n, off, need, offs = len(buf), 0, max_pkts - total, []
            while off + 16 <= n and len(offs) < need:
                incl = u4(buf, off + 8)[0]
                if off + 16 + incl > n:         # 패킷이 chunk 끝에서 잘렸으면 다음 chunk에서 이어서
                    break
                offs.append(off)
                off += 16 + incl
            carry = buf[off:]
            if offs:
                pieces.append(_extract(buf, np.array(offs, dtype=np.int64), dt4))
                total += len(offs)

    if not pieces:
        return None
    fx = {k: np.concatenate([p[k] for p in pieces]) for k in pieces[0]}
    fx['ts'] = (fx['sec'] - fx['sec'][0]).astype(np.float64) + fx['frac'] / div
    del fx['sec'], fx['frac']
    return fx


def make_windows(fx, W):
    """패킷 W개씩 묶어서 window별 통계 feature + 기기 종류(dev)를 만든다"""
    assert W >= 2
    nw = len(fx['ts']) // W
    if nw == 0:
        return None
    n = nw * W
    R = lambda k: fx[k][:n].reshape(nw, W)

    def nuniq(a):                           # window 안의 서로 다른 값 개수
        s = np.sort(a, axis=1)
        return 1 + (s[:, 1:] != s[:, :-1]).sum(axis=1)

    length, ts = R('length'), R('ts')
    iat = np.diff(ts, axis=1)               # 패킷 사이 시간 간격
    dur = ts[:, -1] - ts[:, 0]
    ttl = R('ttl').astype(np.float32)
    fl = R('flags')

    f = {
        'len_mean': length.mean(1), 'len_std': length.std(1),
        'len_min': length.min(1), 'len_max': length.max(1), 'len_sum': length.sum(1),
        'iat_mean': iat.mean(1), 'iat_std': iat.std(1), 'iat_min': iat.min(1), 'iat_max': iat.max(1),
        'duration': dur, 'pkt_rate': np.minimum(W / np.maximum(dur, 1e-6), 1e7),
        'ttl_mean': ttl.mean(1), 'ttl_std': ttl.std(1),
        'n_tcp': R('is_tcp').sum(1), 'n_udp': R('is_udp').sum(1), 'n_icmp': R('is_icmp').sum(1),
        'n_arp': R('is_arp').sum(1), 'n_ipv6': R('is_ip6').sum(1), 'n_frag': R('is_frag').sum(1),
        'n_fin': ((fl & 1) != 0).sum(1), 'n_syn': ((fl & 2) != 0).sum(1), 'n_rst': ((fl & 4) != 0).sum(1),
        'n_psh': ((fl & 8) != 0).sum(1), 'n_ack': ((fl & 16) != 0).sum(1), 'n_urg': ((fl & 32) != 0).sum(1),
        'u_sport': nuniq(R('sport')), 'u_dport': nuniq(R('dport')),
        'u_src_ip': nuniq(R('src_ip')), 'u_dst_ip': nuniq(R('dst_ip')),
        'u_src_mac': nuniq(R('src_mac')), 'u_dst_mac': nuniq(R('dst_mac')),
    }
    df = pd.DataFrame({k: np.asarray(v, dtype=np.float32) for k, v in f.items()})

    # window의 기기 종류: 패킷의 도착지/출발지 MAC 중 IoT 기기(공격자 라즈베리파이 제외)로 가장 많이 나온 종류
    d, s = mac_to_cat(R('dst_mac')), mac_to_cat(R('src_mac'))
    pk = np.where((d > 0) & (d != NEXTGEN), d, np.where((s > 0) & (s != NEXTGEN), s, 0))
    cnt = np.stack([(pk == c).sum(1) for c in range(1, NEXTGEN)], axis=1)
    df['dev'] = np.where(cnt.max(1) > 0, cnt.argmax(1) + 1, 0).astype(np.int8)
    df['wi'] = np.arange(nw)                # 파일 안에서 window 순서(시간 순서)
    return df

print('함수 준비 완료')

함수 준비 완료


## 4) 모든 pcap에서 feature 뽑기
파일마다 결과를 드라이브 `rf_cache` 폴더에 저장해서, 끊겨도 다시 돌리면 끝난 파일은 건너뛰어.

In [4]:
import re

CACHE = PCAP_DIR / 'rf_cache'
CACHE.mkdir(exist_ok=True)

files = [p for p in sorted(PCAP_DIR.glob('**/*.pcap')) if 'rf_cache' not in p.parts]
print(len(files), '개 pcap 파일 발견')

frames = []
for p in files:
    name = p.name
    attack = 'Benign' if 'benign' in name.lower() else re.sub(r'\d*\.pcap$', '', name).rstrip('-')
    cache_f = CACHE / f'{name}_N{MAX_PKTS}_W{WINDOW}.pkl'    # 한 번 뽑은 건 저장해뒀다가 재사용
    if cache_f.exists():
        df = pd.read_pickle(cache_f)
        print(f'[캐시] {name}: window {len(df):,}개')
    else:
        t = time.time()
        fx = read_pcap_fields(p, MAX_PKTS)
        df = make_windows(fx, WINDOW) if fx is not None else None
        if df is None:
            print(f'[건너뜀] {name}: 패킷이 너무 적음')
            continue
        n_pkts, ip_share = len(fx['ts']), (fx['proto'] > 0).mean()
        del fx                                             # 메모리 비우기
        df['file'], df['attack'] = name, attack
        df.to_pickle(cache_f)
        print(f'[완료] {name}: 패킷 {n_pkts:,}개 → window {len(df):,}개 (IPv4 패킷 {ip_share:.0%}, {time.time()-t:.0f}초)')
    frames.append(df)

assert frames, 'pcap 파일을 못 찾았어. 1번 셀의 PCAP_DIR 경로를 확인해줘'
data = pd.concat(frames, ignore_index=True)
print('\n전체 window:', f'{len(data):,}', '/ feature 열:', data.shape[1])

13 개 pcap 파일 발견
[완료] BenignTraffic.pcap: 패킷 2,000,000개 → window 200,000개 (IPv4 패킷 96%, 18초)
[완료] DDoS-ACK_Fragmentation.pcap: 패킷 2,000,000개 → window 200,000개 (IPv4 패킷 99%, 23초)
[완료] DDoS-HTTP_Flood-.pcap: 패킷 2,000,000개 → window 200,000개 (IPv4 패킷 99%, 8초)
[완료] DDoS-ICMP_Flood.pcap: 패킷 2,000,000개 → window 200,000개 (IPv4 패킷 100%, 9초)
[완료] DDoS-ICMP_Fragmentation.pcap: 패킷 2,000,000개 → window 200,000개 (IPv4 패킷 98%, 20초)
[완료] DDoS-PSHACK_Flood.pcap: 패킷 2,000,000개 → window 200,000개 (IPv4 패킷 100%, 6초)
[완료] DDoS-RSTFINFlood.pcap: 패킷 2,000,000개 → window 200,000개 (IPv4 패킷 100%, 8초)
[완료] DDoS-SYN_Flood.pcap: 패킷 2,000,000개 → window 200,000개 (IPv4 패킷 100%, 6초)
[완료] DDoS-SlowLoris.pcap: 패킷 2,000,000개 → window 200,000개 (IPv4 패킷 98%, 11초)
[완료] DDoS-SynonymousIP_Flood.pcap: 패킷 2,000,000개 → window 200,000개 (IPv4 패킷 100%, 6초)
[완료] DDoS-TCP_Flood.pcap: 패킷 2,000,000개 → window 200,000개 (IPv4 패킷 100%, 8초)
[완료] DDoS-UDP_Flood.pcap: 패킷 2,000,000개 → window 200,000개 (IPv4 패킷 100%, 6초)
[완료] DDoS-UDP_Fragmentation.

## 5) 라벨 만들기 + 학습/테스트 나누기 + 기기 종류별 window 수 확인

In [5]:
data['device'] = pd.Categorical.from_codes(data['dev'].astype(int), CATS)

if (data['attack'] == 'Benign').any():
    data['y'] = (data['attack'] != 'Benign').astype(int)      # 0 = 정상, 1 = DDoS
    print('정상(Benign) 데이터가 있어서 → "정상 vs DDoS" 이진 분류로 진행')
else:
    data['y'] = data['attack'].astype('category').cat.codes   # 공격 유형 번호
    print('⚠️ 정상(Benign) pcap이 폴더에 없어서 → "DDoS 공격 유형 구분"(다중 분류)으로 진행')
    print('   BenignTraffic.pcap을 폴더에 넣고 이 셀부터 다시 돌리면 정상 vs DDoS로 바뀌어')

# 파일마다 시간 순서로 앞 70%는 학습, 뒤 30%는 테스트 (이웃한 window가 학습/테스트에 섞이는 걸 막으려고)
data['n_in_file'] = data.groupby('file')['wi'].transform('max') + 1
data['is_train'] = data['wi'] < data['n_in_file'] * 0.7

print('\n[기기 종류별 window 수]  (Unknown = 표에 없는 MAC만 있는 window)')
vc = data['device'].value_counts()
print(vc[vc > 0].to_string())
print('\n[라벨별 window 수]')
print(data['attack'].value_counts().to_string())

정상(Benign) 데이터가 있어서 → "정상 vs DDoS" 이진 분류로 진행

[기기 종류별 window 수]  (Unknown = 표에 없는 MAC만 있는 window)
device
Audio              1762407
Camera              544348
Hub                 210329
Unknown              69512
Home Automation       6446
Power Outlet          5706
Lighting              1252

[라벨별 window 수]
attack
Benign                     200000
DDoS-ACK_Fragmentation     200000
DDoS-HTTP_Flood            200000
DDoS-ICMP_Flood            200000
DDoS-ICMP_Fragmentation    200000
DDoS-PSHACK_Flood          200000
DDoS-RSTFINFlood           200000
DDoS-SYN_Flood             200000
DDoS-SlowLoris             200000
DDoS-SynonymousIP_Flood    200000
DDoS-TCP_Flood             200000
DDoS-UDP_Flood             200000
DDoS-UDP_Fragmentation     200000


## 6) RF 실험: 전체 vs 카메라만

In [6]:
import pickle
assert TARGET_CAT in CATS[1:7], f'TARGET_CAT은 {CATS[1:7]} 중 하나여야 해'
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

DROP = ['file', 'attack', 'dev', 'device', 'wi', 'n_in_file', 'is_train', 'y']
FEATURES = [c for c in data.columns if c not in DROP]      # IP/MAC 값 자체는 feature로 쓰지 않음(개수만 사용)
print('사용 feature 수:', len(FEATURES))

rng = np.random.default_rng(42)
is_tr = data['is_train'].values
is_cat = (data['device'] == TARGET_CAT).values
def pick(mask, n):                          # 조건에 맞는 window 중 최대 n개를 무작위로 뽑기
    idx = np.flatnonzero(mask)
    if len(idx) > n:
        idx = rng.choice(idx, n, replace=False)
    return data.iloc[idx]

tr_all, te_all = pick(is_tr, MAX_TRAIN), pick(~is_tr, MAX_TEST)
tr_cat, te_cat = pick(is_tr & is_cat, MAX_TRAIN), pick(~is_tr & is_cat, MAX_TEST)

print(f'\n전체   : 학습 {len(tr_all):,} / 테스트 {len(te_all):,}   라벨 분포(학습) {tr_all["y"].value_counts().to_dict()}')
print(f'{TARGET_CAT}: 학습 {len(tr_cat):,} / 테스트 {len(te_cat):,}   라벨 분포(학습) {tr_cat["y"].value_counts().to_dict()}')

def train_rf(df):
    rf = RandomForestClassifier(n_estimators=N_TREES, max_depth=MAX_DEPTH, class_weight='balanced',
                                n_jobs=-1, random_state=42)
    t = time.perf_counter()
    rf.fit(df[FEATURES], df['y'])
    train_s = time.perf_counter() - t
    size_mb = len(pickle.dumps(rf)) / 1e6
    rf.set_params(n_jobs=1)                # 추론 속도는 CPU 1코어 기준으로 잼
    return rf, train_s, size_mb

def evaluate(name, rf, train_df, train_s, size_mb, test_df):
    t = time.perf_counter()
    pred = rf.predict(test_df[FEATURES])
    inf_s = time.perf_counter() - t
    return {
        '실험': name, '학습 window 수': len(train_df), '테스트 window 수': len(test_df),
        'Accuracy': round(accuracy_score(test_df['y'], pred), 4),
        'F1(macro)': round(f1_score(test_df['y'], pred, average='macro', zero_division=0), 4),
        '학습시간(초)': round(train_s, 1),
        '추론시간(ms/1000 window)': round(inf_s / len(test_df) * 1e6, 1),
        '모델크기(MB)': round(size_mb, 1),
    }

rows = []
print('\n[1] 전체 데이터로 RF 학습 중...')
rf_all, t_all, s_all = train_rf(tr_all)
rows.append(evaluate('전체 학습 → 전체 테스트', rf_all, tr_all, t_all, s_all, te_all))

if len(tr_cat) > 0 and len(te_cat) > 0:
    rows.append(evaluate(f'전체 학습 → {TARGET_CAT} 테스트', rf_all, tr_all, t_all, s_all, te_cat))
    print(f'[2] {TARGET_CAT} 데이터만으로 RF 학습 중...')
    rf_cat, t_cat, s_cat = train_rf(tr_cat)
    rows.append(evaluate(f'{TARGET_CAT} 학습 → {TARGET_CAT} 테스트', rf_cat, tr_cat, t_cat, s_cat, te_cat))
else:
    print(f'⚠️ {TARGET_CAT} window가 학습 또는 테스트에 없어서 {TARGET_CAT} 실험은 건너뜀 (위 기기 종류별 window 수를 확인해봐)')

res = pd.DataFrame(rows)
res.to_csv(PCAP_DIR / 'rf_full_vs_camera_result.csv', index=False, encoding='utf-8-sig')
print('\n===== 결과 (표는 드라이브에도 저장됨: rf_full_vs_camera_result.csv) =====')
res

사용 feature 수: 31

전체   : 학습 300,000 / 테스트 200,000   라벨 분포(학습) {1: 277065, 0: 22935}
Camera: 학습 300,000 / 테스트 101,686   라벨 분포(학습) {1: 257585, 0: 42415}

[1] 전체 데이터로 RF 학습 중...
[2] Camera 데이터만으로 RF 학습 중...

===== 결과 (표는 드라이브에도 저장됨: rf_full_vs_camera_result.csv) =====


,실험,학습 window 수,테스트 window 수,Accuracy,F1(macro),학습시간(초),추론시간(ms/1000 window),모델크기(MB)
0,전체 학습 → 전체 테스트,300000,200000,0.9726,0.9155,23.8,5.1,15.2
1,전체 학습 → Camera 테스트,300000,101686,0.8700,0.8700,23.8,7.5,15.2
2,Camera 학습 → Camera 테스트,300000,101686,0.8837,0.8836,40.6,7.4,37.7


## 결과 읽는 법
- **1행**: 전체 데이터로 학습한 RF의 전체 성능
- **2행**: 그 모델을 카메라 window에서만 평가한 성능 (카메라 트래픽에서도 잘 맞히는지)
- **3행**: 카메라 window만으로 새로 학습한 RF. 1행과 정확도가 비슷한데 학습시간·모델크기가 줄었으면 "카메라만으로도 충분"이라고 말할 수 있어
- **추론시간**은 CPU 1코어로 window 1000개를 예측하는 데 걸린 시간(ms)
- 트리 수·깊이는 세 실험에서 똑같아서, 속도 차이는 주로 학습 window 수 차이에서 나와